# Discovery Platform - GPU Worker (Colab)

Free GPU compute for intensive analysis tasks

## Setup
1. Runtime > Change runtime type > GPU (T4)
2. Run all cells
3. Copy the ngrok URL to your `.env` file

**Free Tier:**
- GPU: NVIDIA T4 (16GB VRAM)
- RAM: 12GB
- Disk: 78GB
- Runtime: 12 hours max (resets daily)

In [ ]:
# Install dependencies
!pip install -q fastapi uvicorn pyngrok nest-asyncio numpy pandas

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Setup ngrok for public access
from pyngrok import ngrok
import getpass

# Get ngrok auth token (sign up free at https://dashboard.ngrok.com/signup)
ngrok_token = getpass.getpass("Enter your ngrok auth token: ")
ngrok.set_auth_token(ngrok_token)

In [ ]:
# Worker API
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Any
import numpy as np
import pandas as pd
import uvicorn
import nest_asyncio
from threading import Thread

nest_asyncio.apply()

app = FastAPI(
    title="Discovery GPU Worker",
    description="GPU-accelerated worker running on Google Colab",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class AnalysisRequest(BaseModel):
    trades: List[Dict[str, Any]]
    analysis_types: List[str]
    options: Dict[str, Any] = {}

@app.get("/")
async def root():
    return {
        "service": "Discovery GPU Worker",
        "platform": "Google Colab",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
        "status": "running"
    }

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "worker_type": "gpu",
        "compute_platform": "colab",
        "gpu_available": torch.cuda.is_available(),
        "available_analyses": [
            "sentiment",
            "volume_analysis",
            "price_patterns",
            "ml_predictions",
            "clustering"
        ]
    }

@app.post("/analyze")
async def analyze(request: AnalysisRequest):
    try:
        results = {}
        
        # Convert trades to DataFrame for easier processing
        df = pd.DataFrame(request.trades)
        
        if "sentiment" in request.analysis_types:
            results["sentiment"] = analyze_sentiment_gpu(df)
        
        if "volume_analysis" in request.analysis_types:
            results["volume"] = analyze_volume_gpu(df)
        
        if "price_patterns" in request.analysis_types:
            results["patterns"] = analyze_patterns_gpu(df)
        
        if "ml_predictions" in request.analysis_types:
            results["predictions"] = ml_predictions_gpu(df)
        
        return {
            "status": "success",
            "worker_id": "colab-gpu-worker-1",
            "gpu_used": torch.cuda.is_available(),
            "results": results,
            "trades_processed": len(request.trades)
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def analyze_sentiment_gpu(df):
    """GPU-accelerated sentiment analysis"""
    if 'price_change' not in df.columns:
        return {"error": "Missing price_change column"}
    
    # Use GPU for calculations if available
    if torch.cuda.is_available():
        prices = torch.tensor(df['price_change'].values, device='cuda')
        positive = (prices > 0).sum().item()
        negative = (prices < 0).sum().item()
    else:
        positive = (df['price_change'] > 0).sum()
        negative = (df['price_change'] < 0).sum()
    
    return {
        "positive": int(positive),
        "negative": int(negative),
        "neutral": len(df) - positive - negative,
        "overall_sentiment": "positive" if positive > negative else "negative"
    }

def analyze_volume_gpu(df):
    """GPU-accelerated volume analysis"""
    if 'volume' not in df.columns:
        return {"error": "Missing volume column"}
    
    if torch.cuda.is_available():
        volumes = torch.tensor(df['volume'].values, device='cuda', dtype=torch.float32)
        return {
            "average_volume": float(volumes.mean().item()),
            "total_volume": float(volumes.sum().item()),
            "max_volume": float(volumes.max().item()),
            "min_volume": float(volumes.min().item()),
            "std_volume": float(volumes.std().item())
        }
    else:
        return {
            "average_volume": float(df['volume'].mean()),
            "total_volume": float(df['volume'].sum()),
            "max_volume": float(df['volume'].max()),
            "min_volume": float(df['volume'].min()),
            "std_volume": float(df['volume'].std())
        }

def analyze_patterns_gpu(df):
    """GPU-accelerated pattern detection"""
    if 'price' not in df.columns:
        return {"error": "Missing price column"}
    
    if torch.cuda.is_available():
        prices = torch.tensor(df['price'].values, device='cuda', dtype=torch.float32)
        returns = (prices[1:] - prices[:-1]) / prices[:-1]
        
        return {
            "pattern": "uptrend" if returns.mean() > 0 else "downtrend",
            "volatility": float(returns.std().item()),
            "avg_return": float(returns.mean().item()),
            "max_drawdown": float(returns.min().item())
        }
    else:
        returns = df['price'].pct_change().dropna()
        return {
            "pattern": "uptrend" if returns.mean() > 0 else "downtrend",
            "volatility": float(returns.std()),
            "avg_return": float(returns.mean())
        }

def ml_predictions_gpu(df):
    """Simple ML predictions (placeholder for more advanced models)"""
    return {
        "model": "simple_trend",
        "prediction": "hold",
        "confidence": 0.75,
        "note": "Upgrade to more sophisticated models as needed"
    }

print("\n✅ Worker API configured successfully!")

In [ ]:
# Start the server
from pyngrok import ngrok

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print("\n" + "="*70)
print("🚀 WORKER IS LIVE!")
print("="*70)
print(f"\n📍 Public URL: {public_url}")
print(f"\n💡 Add this to your .env file:")
print(f"   COLAB_GPU_WORKER={public_url}")
print(f"\n🧪 Test with:")
print(f"   curl {public_url}/health")
print("\n" + "="*70 + "\n")

# Run the server
uvicorn.run(app, host="0.0.0.0", port=8000)

## Usage Notes

**Runtime Limits:**
- Max 12 hours per session
- Auto-disconnects after ~90 min idle
- Need to restart daily

**Best For:**
- Heavy ML model inference
- Large batch processing
- Matrix operations
- Deep learning tasks

**Not For:**
- 24/7 services (use Oracle/HuggingFace)
- Database operations
- Long-running background jobs

**Tips:**
- Keep browser tab open
- Run lightweight tasks on other workers
- Save checkpoints frequently
- Use for GPU-specific workloads only